# Sentinel-1 SAR Flood Detection — Sandbox

Goal: detect flood extent in Sylhet district during the 2024 monsoon
flood event using Sentinel-1 VV-polarization backscatter thresholding,
masked against JRC Global Surface Water permanent-water layer.

Validation case: Sylhet floods, May–June 2024.

In [3]:
%pip install --quiet folium streamlit-folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import ee
import pandas as pd
import folium

ee.Initialize(project="earth-engine-project-495404")
print("Earth Engine ready")
print("folium version:", folium.__version__)

Earth Engine ready
folium version: 0.20.0


In [5]:
districts = (
    ee.FeatureCollection("FAO/GAUL/2015/level2")
    .filter(ee.Filter.eq("ADM0_NAME", "Bangladesh"))
)

sylhet = districts.filter(ee.Filter.eq("ADM2_NAME", "Sylhet")).first()
sylhet_geom = sylhet.geometry()

area_km2 = sylhet_geom.area().divide(1e6).getInfo()
centroid = sylhet_geom.centroid().coordinates().getInfo()

print(f"Sylhet area: {area_km2:.1f} km^2")
print(f"Sylhet centroid (lon, lat): {centroid[0]:.4f}, {centroid[1]:.4f}")

Sylhet area: 3481.8 km^2
Sylhet centroid (lon, lat): 91.9933, 24.9175


In [6]:
flood_start = ee.Date("2024-05-25")
flood_end = ee.Date("2024-06-30")

pre_start = ee.Date("2024-05-01")
pre_end = ee.Date("2024-05-20")

s1_base = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(sylhet_geom)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
)

pre = s1_base.filterDate(pre_start, pre_end)
flood = s1_base.filterDate(flood_start, flood_end)

print("Pre-flood window (May 1-20, 2024):")
print(f"  Scene count: {pre.size().getInfo()}")
print(f"  Orbit passes: {pre.aggregate_array('orbitProperties_pass').distinct().getInfo()}")

print("\nFlood window (May 25 - Jun 30, 2024):")
print(f"  Scene count: {flood.size().getInfo()}")
print(f"  Orbit passes: {flood.aggregate_array('orbitProperties_pass').distinct().getInfo()}")

Pre-flood window (May 1-20, 2024):
  Scene count: 3
  Orbit passes: ['DESCENDING', 'ASCENDING']

Flood window (May 25 - Jun 30, 2024):
  Scene count: 6
  Orbit passes: ['DESCENDING', 'ASCENDING']


In [7]:
ORBIT_PASS = "DESCENDING"

pre_d = pre.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))
flood_d = flood.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))

print(f"After filtering to {ORBIT_PASS}:")
print(f"  Pre-flood scenes: {pre_d.size().getInfo()}")
print(f"  Flood scenes: {flood_d.size().getInfo()}")

pre_composite = pre_d.select("VV").median().clip(sylhet_geom)
flood_composite = flood_d.select("VV").median().clip(sylhet_geom)

print("\nMedian composites built (no client-side computation yet)")

After filtering to DESCENDING:
  Pre-flood scenes: 2
  Flood scenes: 3

Median composites built (no client-side computation yet)


In [8]:
THRESHOLD_DB = -16

pre_water = pre_composite.lt(THRESHOLD_DB).rename("water")
flood_water = flood_composite.lt(THRESHOLD_DB).rename("water")

def water_area_km2(water_mask):
    area_image = (
        water_mask
        .multiply(ee.Image.pixelArea())
        .divide(1e6)
        .rename("area")
    )
    return area_image.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=sylhet_geom,
        scale=30,
        maxPixels=1e9,
    ).get("area").getInfo()

pre_area = water_area_km2(pre_water)
flood_area = water_area_km2(flood_water)

print(f"Pre-flood water area:    {pre_area:.1f} km^2")
print(f"Flood-period water area: {flood_area:.1f} km^2")
print(f"Increase:                {flood_area - pre_area:+.1f} km^2")
print(f"Multiplier:              {flood_area / pre_area:.1f}x")

Pre-flood water area:    193.5 km^2
Flood-period water area: 1448.6 km^2
Increase:                +1255.1 km^2
Multiplier:              7.5x


In [9]:
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
permanent_water = gsw.gt(50).unmask(0).rename("water")

flood_only = flood_water.updateMask(permanent_water.eq(0)).rename("water")
intersection = flood_water.multiply(permanent_water).rename("water")

permanent_area = water_area_km2(permanent_water.clip(sylhet_geom))
intersection_area = water_area_km2(intersection)
flood_only_area = water_area_km2(flood_only)
district_area = 3481.8

print(f"Permanent water in Sylhet:        {permanent_area:.1f} km^2")
print(f"Flood AND permanent (overlap):    {intersection_area:.1f} km^2")
print(f"Flood-only extent:                {flood_only_area:.1f} km^2")
print(f"Flood-only as share of district:  {flood_only_area / district_area * 100:.1f}%")

Permanent water in Sylhet:        102.9 km^2
Flood AND permanent (overlap):    99.6 km^2
Flood-only extent:                1301.5 km^2
Flood-only as share of district:  37.4%


In [10]:
def ee_tile_layer(image, vis_params, name, opacity=0.7):
    map_id = image.getMapId(vis_params)
    return folium.TileLayer(
        tiles=map_id["tile_fetcher"].url_format,
        attr="Google Earth Engine",
        name=name,
        overlay=True,
        control=True,
        opacity=opacity,
    )

m = folium.Map(
    location=[centroid[1], centroid[0]],
    zoom_start=9,
    tiles="OpenStreetMap",
)

ee_tile_layer(
    permanent_water.selfMask(),
    {"palette": ["#1565C0"]},
    "Permanent water (JRC)",
    opacity=0.6,
).add_to(m)

ee_tile_layer(
    flood_only.selfMask(),
    {"palette": ["#D32F2F"]},
    "Flood extent (May 25 - Jun 30, 2024)",
    opacity=0.7,
).add_to(m)

folium.GeoJson(
    sylhet_geom.getInfo(),
    style_function=lambda feature: {
        "color": "black",
        "weight": 2,
        "fillOpacity": 0,
    },
    name="Sylhet boundary",
).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

m

In [11]:
control_pre_start = ee.Date("2024-02-01")
control_pre_end = ee.Date("2024-02-20")
control_target_start = ee.Date("2024-02-25")
control_target_end = ee.Date("2024-03-31")

ctrl_pre = s1_base.filterDate(control_pre_start, control_pre_end)
ctrl_pre_d = ctrl_pre.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))
ctrl_target = s1_base.filterDate(control_target_start, control_target_end)
ctrl_target_d = ctrl_target.filter(ee.Filter.eq("orbitProperties_pass", ORBIT_PASS))

print(f"Control pre scenes:    {ctrl_pre_d.size().getInfo()}")
print(f"Control target scenes: {ctrl_target_d.size().getInfo()}")

ctrl_pre_composite = ctrl_pre_d.select("VV").median().clip(sylhet_geom)
ctrl_target_composite = ctrl_target_d.select("VV").median().clip(sylhet_geom)

ctrl_pre_water = ctrl_pre_composite.lt(THRESHOLD_DB).rename("water")
ctrl_target_water = ctrl_target_composite.lt(THRESHOLD_DB).rename("water")
ctrl_target_only = ctrl_target_water.updateMask(permanent_water.eq(0)).rename("water")

ctrl_pre_area = water_area_km2(ctrl_pre_water)
ctrl_target_area = water_area_km2(ctrl_target_water)
ctrl_target_only_area = water_area_km2(ctrl_target_only)

print(f"\nDry-season control (Feb-Mar 2024):")
print(f"  Pre water:        {ctrl_pre_area:.1f} km^2")
print(f"  Target water:     {ctrl_target_area:.1f} km^2")
print(f"  'Flood-only':     {ctrl_target_only_area:.1f} km^2")
print(f"  Share of district: {ctrl_target_only_area / district_area * 100:.1f}%")

Control pre scenes:    1
Control target scenes: 3

Dry-season control (Feb-Mar 2024):
  Pre water:        127.9 km^2
  Target water:     75.1 km^2
  'Flood-only':     27.9 km^2
  Share of district: 0.8%
